In [1]:
# --- Notebook Test Engine: injected setup (auto-generated) ---
import os as _os
for _k in ('AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_SESSION_TOKEN','AWS_CREDENTIAL_EXPIRATION'):
    _os.environ.pop(_k, None)
_os.environ['AWS_DEFAULT_REGION'] = 'us-west-2'
_os.environ['AWS_REGION'] = 'us-west-2'
_nte_role = 'arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole'
try:
    from sagemaker.core.helper import session_helper as _sh
    _sh.get_execution_role = lambda *a, **k: _nte_role
except Exception:
    pass
try:
    _nte_cf = _os.environ.get('AWS_SHARED_CREDENTIALS_FILE')
    if _nte_cf and _os.path.exists(_nte_cf):
        import configparser as _cfp, datetime as _dt
        import boto3 as _b3, botocore.session as _bcs
        from botocore.credentials import RefreshableCredentials as _RC
        _nte_prof = _os.environ.get('AWS_PROFILE', 'default')
        def _nte_load():
            _cp = _cfp.ConfigParser(); _cp.read(_nte_cf)
            _s = _cp[_nte_prof]
            _tok = _s.get('aws_session_token')
            if not _tok:
                raise RuntimeError('nte: no session token in creds file')
            return {'access_key': _s['aws_access_key_id'],
                    'secret_key': _s['aws_secret_access_key'],
                    'token': _tok,
                    'expiry_time': (_dt.datetime.now(_dt.timezone.utc)
                                    + _dt.timedelta(minutes=9)).isoformat()}
        _nte_creds = _RC.create_from_metadata(metadata=_nte_load(),
                                              refresh_using=_nte_load,
                                              method='nte-shared-file')
        def _nte_get_credentials(self, *a, **k):
            # Honor sessions that were handed explicit credentials
            # (e.g. notebooks that assume_role into other accounts):
            # clobbering them would silently run every cross-account
            # client as the ambient identity. Bare/SDK-created sessions
            # (no explicit creds) still get the refreshable shared-file
            # creds so long-running waits survive credential rotation.
            _ex = getattr(self, '_credentials', None)
            if _ex is not None:
                return _ex
            return _nte_creds
        _bcs.Session.get_credentials = _nte_get_credentials
        _b3.setup_default_session(botocore_session=_bcs.get_session())
        print('nte: refreshable shared-file creds installed')
except Exception as _e:
    print('nte: refreshable-creds shim skipped:', _e)
try:
    from sagemaker.core.resources import ModelPackageGroup as _NteMPG
    _nte_mpg_create = _NteMPG.create
    def _nte_mpg_get_or_create(*_a, **_k):
        try:
            return _nte_mpg_create(*_a, **_k)
        except Exception as _e2:
            if 'already exists' in str(_e2).lower():
                _name = _k.get('model_package_group_name') or (_a[0] if _a else None)
                return _NteMPG.get(_name)
            raise
    _NteMPG.create = staticmethod(_nte_mpg_get_or_create)
    print('nte: ModelPackageGroup.create is now get-or-create')
except Exception as _e:
    print('nte: ModelPackageGroup idempotency shim skipped:', _e)


nte: ModelPackageGroup.create is now get-or-create


# SageMaker Custom Scorer Evaluation - Demo

This notebook demonstrates how to use the CustomScorerEvaluator to evaluate models with custom evaluator functions.

## Setup

Import necessary modules.

In [2]:
# Configure AWS credentials and region
#! ada credentials update --provider=isengard --account=<> --role=Admin --profile=default --once
#! aws configure set region us-west-2

In [3]:
from sagemaker.train.evaluate import CustomScorerEvaluator
from rich.pretty import pprint

# Configure logging to show INFO messages
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s - %(name)s - %(message)s'
)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


## Configure Evaluation Parameters

Set up the parameters for your custom scorer evaluation.

In [4]:
# Evaluator ARN (custom evaluator from AI Registry)
# evaluator_arn = "arn:aws:sagemaker:us-west-2:<>:hub-content/AIRegistry/JsonDoc/00-goga-qa-evaluation/1.0.0"
# evaluator_arn = "arn:aws:sagemaker:us-west-2:<>:hub-content/AIRegistry/JsonDoc/nikmehta-reward-function/1.0.0"
# evaluator_arn = "arn:aws:sagemaker:us-west-2:<>:hub-content/AIRegistry/JsonDoc/eval-lambda-test/0.0.1"
evaluator_arn = "prime_math"

# Dataset - can be S3 URI or AIRegistry DataSet ARN
dataset = "s3://notebook-test-engine-ds-674622101542-usw2/datasets/sample-eval-prompts.jsonl"

# Base model - can be:
# 1. Model package ARN: "arn:aws:sagemaker:region:account:model-package/name/version"
# 2. JumpStart model ID: "llama-3-2-1b-instruct" [Evaluation with Base Model Only is yet to be implemented/tested - Not Working currently]
base_model = "arn:aws:sagemaker:us-west-2:674622101542:model-package/sdk-test-finetuned-models/3"

# S3 location for outputs
s3_output_path = "s3://notebook-test-engine-ds-674622101542-usw2/output/custom-scorer-eval/"

# Optional: MLflow tracking server ARN
mlflow_resource_arn = None

print("Configuration:")
print(f"  Evaluator: {evaluator_arn}")
print(f"  Dataset: {dataset}")
print(f"  Base Model: {base_model}")
print(f"  Output Location: {s3_output_path}")

Configuration:
  Evaluator: prime_math
  Dataset: s3://notebook-test-engine-ds-674622101542-usw2/datasets/sample-eval-prompts.jsonl
  Base Model: arn:aws:sagemaker:us-west-2:674622101542:model-package/sdk-test-finetuned-models/3
  Output Location: s3://notebook-test-engine-ds-674622101542-usw2/output/custom-scorer-eval/


## Create CustomScorerEvaluator Instance

Instantiate the evaluator with your configuration. The evaluator can accept:
- **Custom Evaluator ARN** (string): Points to your custom evaluator in AI Registry
- **Built-in Metric** (string or enum): Use preset metrics like "code_executions", "math_answers", etc.
- **Evaluator Object**: A sagemaker.ai_registry.evaluator.Evaluator instance

In [5]:
# Create evaluator with custom evaluator ARN
evaluator = CustomScorerEvaluator(role="arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole", 
    evaluator=evaluator_arn,  # Custom evaluator ARN
    dataset=dataset,
    model=base_model,
    s3_output_path=s3_output_path,
    mlflow_resource_arn=mlflow_resource_arn,
    # model_package_group="arn:aws:sagemaker:us-west-2:<>:model-package-group/Demo-test-deb-2", 
    evaluate_base_model=False  # Set to True to also evaluate the base model
)

print("\n✓ CustomScorerEvaluator created successfully")
pprint(evaluator)

[07/21/26 23:27:47] INFO     Runs on sagemaker prod, region:us-west-2                                  ]8;id=3562940;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=3562941;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py#375\375]8;;\

                    INFO     Retrieved ModelPackage in region: us-west-2                    ]8;id=3562948;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/model_resolution.py\model_resolution.py]8;;\:]8;id=3562949;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/model_resolution.py#425\425]8;;\

                    INFO     Found 1 MLflow apps: [('finetune-mlflow-1783049203', 'Created',  ]8;id=3562956;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=3562957;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/finetune_utils.py#213\213]8;;\
                             '3.10.1')]                                                                            

[07/21/26 23:27:48] INFO     Resolved MLflow app:                                             ]8;id=3562963;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=3562964;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/finetune_utils.py#236\236]8;;\
                             arn:aws:sagemaker:us-west-2:674622101542:mlflow-app/app-3THB6MFI                      
                             DE4F (status: Created, version: 3.10.1)                                               

                    INFO     Resolved MLflow resource ARN:                                    ]8;id=3562971;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=3562972;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py#215\215]8;;\
                             arn:aws:sagemaker:us-west-2:674622101542:mlflow-app/app-3THB6MFI                      
                             DE4F                                                                                  


✓ CustomScorerEvaluator created successfully


CustomScorerEvaluator(
│   region=None,
│   role='arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole',
│   sagemaker_session=<sagemaker.core.helper.session_helper.Session object at 0x7f4d4262ecf0>,
│   model='arn:aws:sagemaker:us-west-2:674622101542:model-package/sdk-test-finetuned-models/3',
│   base_model_name=None,
│   base_eval_name='eval-meta-5669fe32',
│   s3_output_path='s3://notebook-test-engine-ds-674622101542-usw2/output/custom-scorer-eval/',
│   mlflow_resource_arn='arn:aws:sagemaker:us-west-2:674622101542:mlflow-app/app-3THB6MFIDE4F',
│   mlflow_experiment_name=None,
│   mlflow_run_name=None,
│   networking=None,
│   kms_key_id=None,
│   model_package_group=None,
│   compute=None,
│   training_image=None,
│   recipe=None,
│   overrides=None,
│   evaluator=<_BuiltInMetric.PRIME_MATH: 'prime_math'>,
│   dataset='s3://notebook-test-engine-ds-674622101542-usw2/datasets/sample-eval-prompts.jsonl',
│   evaluate_base_model=False
)

### Optionally update the hyperparameters

In [6]:
pprint(evaluator.hyperparameters.to_dict())

# optionally update hyperparameters
# evaluator.hyperparameters.temperature = "0.1"

# optionally get more info on types, limits, defaults.
# evaluator.hyperparameters.get_info()

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=3562979;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=3562980;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     Retrieved ModelPackage in region: us-west-2                    ]8;id=3562985;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/model_resolution.py\model_resolution.py]8;;\:]8;id=3562986;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/model_resolution.py#425\425]8;;\

                    INFO     Fetching evaluation override parameters for             ]8;id=3562993;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/custom_scorer_evaluator.py\custom_scorer_evaluator.py]8;;\:]8;id=3562994;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/custom_scorer_evaluator.py#239\239]8;;\
                             hyperparameters property                                                              

                    INFO     Fetching hub content metadata for                                  ]8;id=3563001;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/recipe_utils.py\recipe_utils.py]8;;\:]8;id=3563002;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/recipe_utils.py#226\226]8;;\
                             meta-textgeneration-llama-3-2-1b-instruct from SageMakerPublicHub                     

                    INFO     Searching for evaluation recipe with Type='Evaluation' and         ]8;id=3563008;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/recipe_utils.py\recipe_utils.py]8;;\:]8;id=3563009;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/recipe_utils.py#246\246]8;;\
                             EvaluationType='DeterministicEvaluation'                                              

                    INFO     Downloading override parameters from                               ]8;id=3563015;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/recipe_utils.py\recipe_utils.py]8;;\:]8;id=3563016;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/recipe_utils.py#271\271]8;;\
                             s3://jumpstart-cache-prod-us-west-2/recipes/open-source-eval-meta-                    
                             textgeneration-llama-3-2-1b-instruct-deterministic_override_params                    
                             _sm_jobs_v2.0.2.json                                                                  

{
│   'max_new_tokens': '2048',
│   'temperature': '0',
│   'top_k': '-1',
│   'top_p': '1.0',
│   'aggregation': '',
│   'postprocessing': 'False',
│   'max_model_len': '12000'
}

## Alternative: Using Built-in Metrics

Instead of a custom evaluator ARN, you can use built-in metrics:

In [7]:
# Example with built-in metrics (commented out)
# from sagemaker.train.evaluate import get_builtin_metrics
# 
# BuiltInMetric = get_builtin_metrics()
# 
# evaluator_builtin = CustomScorerEvaluator(
#     evaluator=BuiltInMetric.PRIME_MATH,  # Or use string: "prime_math"
#     dataset=dataset,
#     base_model=base_model,
#     s3_output_path=s3_output_path
# )

## Start Evaluation

Call `evaluate()` to start the evaluation job. This will:
1. Create or update the evaluation pipeline
2. Start a pipeline execution
3. Return an `EvaluationPipelineExecution` object for monitoring

In [8]:
# Start evaluation
execution = evaluator.evaluate()

print("\n✓ Evaluation execution started successfully!")
print(f"  Execution Name: {execution.name}")
print(f"  Pipeline Execution ARN: {execution.arn}")
print(f"  Status: {execution.status.overall_status}")

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=3563021;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=3563022;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/21/26 23:27:49] INFO     Cannot simulate policies for                                  ]8;id=3563029;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=3563030;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=3563036;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=3563037;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'training' (see                                                       
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

                    INFO     Getting or creating artifact for source:                         ]8;id=3563043;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=3563044;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py#805\805]8;;\
                             arn:aws:sagemaker:us-west-2:674622101542:model-package/sdk-test-                      
                             finetuned-models/3                                                                    

                    INFO     Searching for existing artifact for model package:               ]8;id=3563050;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=3563051;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py#624\624]8;;\
                             arn:aws:sagemaker:us-west-2:674622101542:model-package/sdk-test-                      
                             finetuned-models/3                                                                    

...

                    INFO     Final Resource Status: Active                                       ]8;id=3563170;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3563171;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#27475\27475]8;;\

                    INFO     Pipeline                                                              ]8;id=3563177;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=3563178;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#122\122]8;;\
                             SagemakerEvaluation-CustomScorerEvaluation-91d7d4da-03ce-4c62-aaa6-aa                 
                             69a81b2b82 is now active and ready for execution                                      

                    INFO     Starting pipeline execution: eval-meta-5669fe32-1784676470            ]8;id=3563184;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=3563185;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#319\319]8;;\

[07/21/26 23:27:51] INFO     Pipeline execution started:                                           ]8;id=3563191;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=3563192;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#330\330]8;;\
                             arn:aws:sagemaker:us-west-2:674622101542:pipeline/SagemakerEvaluation                 
                             -CustomScorerEvaluation-91d7d4da-03ce-4c62-aaa6-aa69a81b2b82/executio                 
                             n/fg3gkadt11wc                                                                        


✓ Evaluation execution started successfully!
  Execution Name: eval-meta-5669fe32
  Pipeline Execution ARN: arn:aws:sagemaker:us-west-2:674622101542:pipeline/SagemakerEvaluation-CustomScorerEvaluation-91d7d4da-03ce-4c62-aaa6-aa69a81b2b82/execution/fg3gkadt11wc
  Status: Executing


## Monitor Job Progress

Use `refresh()` to update the job status, or `wait()` to block until completion.

In [9]:
# Check current status
execution.refresh()
print(f"Current Status: {execution.status.overall_status}")

pprint(execution.status)

[07/21/26 23:27:52] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=3563197;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=3563198;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Current Status: Executing


PipelineExecutionStatus(
│   overall_status='Executing',
│   step_details=[
│   │   StepDetail(
│   │   │   name='CreateEvaluationAction',
│   │   │   status='Starting',
│   │   │   start_time='2026-07-21T23:27:51.800000+00:00',
│   │   │   end_time=None,
│   │   │   display_name=None,
│   │   │   failure_reason=None,
│   │   │   job_arn=None
│   │   ),
│   │   StepDetail(
│   │   │   name='EvaluateCustomModel',
│   │   │   status='Starting',
│   │   │   start_time='2026-07-21T23:27:51.800000+00:00',
│   │   │   end_time=None,
│   │   │   display_name=None,
│   │   │   failure_reason=None,
│   │   │   job_arn=None
│   │   )
│   ],
│   failure_reason=None
)

## Wait for Completion

Block execution until the job completes. This provides a rich visual experience in Jupyter notebooks.

In [10]:
# Wait for job to complete (with rich visual feedback)
execution.wait(poll=30, timeout=3600)

print(f"\nFinal Status: {execution.status.overall_status}")

[07/21/26 23:40:33] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=3563648;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=3563649;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

╭─────────────────────────────────────────── Pipeline Execution Status ───────────────────────────────────────────╮
│  Evaluation Job        eval-meta-5669fe32-1784676470                                                            │
│  Links                 ]8;id=3563655;https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#logsV2:log-groups/log-group/$252Faws$252Fsagemaker$252FTrainingJobs$3FlogStreamNameFilter$3Dpipelines-fg3gkadt11wc-EvaluateCustomModel-zoonTwz5R1\🔗 CloudWatch Logs]8;;\ | ]8;id=3563656;https://app-3THB6MFIDE4F.mlflow.sagemaker.us-west-2.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IkRXU1FVNyIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNFg2b0drWWdUdGNVYUtiVFk4ZUk5MEg1VzlzUkVoUHdLTHJJTWdWZm5yQ0FBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVFMVVXeHJSazVrZG5Kc1ZFOHJRa2gyWlVOelEzZFJNaXQzYld4UVlqTnpjMUpEVjBRM1UwdG9NR2NyTkdkNVNIZERUbUZqWkdZMGMyTmxjMmN4TUV4MmR6MDlBQUVBQjJGM2N5MXJiWE1BUzJGeWJqcGhkM002YTIxek9uVnpMWGRsYzNRdE1qbzVOell4T1RNeU5UUTFOemM2YTJWNUwyVmhNbUZrTUdSa0xXRmtNV0l0TkROalppMDRaak0xTFRNMU5UWmlZVE5pWm1NeE1nQzRBUUlCQUhqRHJwaCtQMVNKcHUzZUl2YnVXUi94bGQ2THMxNjhEU3JZbXo5akp0dlBUQUhwY0FSYUJhWWhiSVdDdUtQQzJOVXNBQUFBZmpCOEJna3Foa2lHOXcwQkJ3YWdiekJ0QWdFQU1HZ0dDU3FHU0liM0RRRUhBVEFlQmdsZ2hrZ0JaUU1FQVM0d0VRUU1WUFFaTEhtNG5qcmV4Z0NxQWdFUWdEdXVlbnUya3pyS2dkd2Fpd0RpRmFCU0N5S1g0Qm1aTUNTOUxBYTAxSkRsVS85bU9iZlI2Rmg5SForckU3RVNLTm43dXI1dFZmL081UWlvVWdJQUFCQUFwOFpkNVlQRnl4UGpXSEM4M0FCdFFpeTdJZ092czVPamY0KzMzdEJkV2ZOZFluNnlZcFNkNGdxbFhkNVdXYnlKLy8vLy93QUFBQUVBQUFBQUFBQUFBQUFBQUFFQUFBUVAyT3ZJYUhuVm0yUEl1QkFSRjF0c0g1YVVVRU1UekdRN3NXbS9LSU9RckZNOUtEOGR3WHowNVloc0tyOXovWC9nTkQyNFhXYlF0cUJFbnVOaTB2L1N6dC9RWURFNXlqWDg4NXlZNDNwazVEK253Zk5VZDZLWFd6dTBVeUh5SGRFM05DUHpXV3c3Ry9oREFyejVYSTZHdWw0U1BwY2FOV2NGWTVkRnZ6MjA5Rlhic1l6a2FINEZFYndlVHFhT25ESXE3eFFNVTdKcVpFVWhjZDVMc0NkcDhmdElQUVNMMjNSc2JyVUg2ZDgzemgyRjVxYjJJTmIvVk5DRnpac0dtbEgxUytFM3RkSjR4Y0UyelBBb2U0dGRMQ0Nvcko1VjdwYTNjS0psL0c0YTduZnZiOUwwRXBQL2ZyTHJXaEg4REJ3QzFPNzN2dndwNDlmSnVPWXcvY29zemdPOUFxcmhsU1lOcHVWUmlUQzZoSDNIRmN1OGhKOGZudzd6djdtS0JSM0N4UUNPOXFvbGxuTTlOUGNjNm1BSTVOMXBqR0tLaGVuN0sxb3h4SUFmbzJSdW9hWTkrdVlTeVBTdlIxUFp1YitwSm5HaGx0QitxcFZ0UUErSXFNeGVDNzFlZGJvQjdXbHpWVnZHWk1LV1lnOEdNUVJ0V2dGS1hsdTdRM1VHQ3U5NEpxSDRlQSsyNFJYYktKQkhjK1VGbFkxNVQ0Z08vRFc1cGp4V3RZN1ZHY1JEejNMZE13MHFtQXp6Rk5zM3lhZ1JINGhBM2tEaTlJSDVDV2U0NTNRQlZOR1Q5SUFhcmI3bkp3SHg1M01YZTBuNXZqSlk4TWpHTE8zenYwODl2UENOanBpN29EVHd6emdmbnFXd1Yxc24zeWpUQVZKRmFsWXQrb0FDUW1xbXEzWjdJeDFVbU1lYjNjUSt1bG9uZ0lFbk5wekFtT2pQYTk2ZmxjSUFtUDgxaTVSRzRaT2U1MUNCaUkzTVVNc1N2RkNzVHhiWnBpL1hsai9WcjIySUUxWVg3c3RNY2RjMS9GQmgrN1l5dzRoRXZpQWZBdzZLUThlcGdCaXdpQTZVY0ZsOUxjM1lLbFlqL2V4bVNZNWY1aXpzNXlLVHJ1aXhiNnE0eDNoV1lSaG5WTG8xSWlxUWN0TDRESkZSTFEySis1Qk8xekxSVlVSN0h5emZOSGRQWFBvNzZ5TUU0TC9lbG5tRHRDY2M2Q3cvUUJQRzZ0U094U1d4U2pjUGNYbnF0Q09UNlYweG1Sc0prMk02djFxd1FGRGhGRjdSRUMxTU9BbXM2clJBMDd2Z21YUmFWWCt5Q3c2S1dWOE1zaFp3WXJxbmdMVnFhMHluQmVmTUo0L1g5bXlvNTZwMjVkekhydE9keGIyVFhvdHhnWitOMU4vTkxmNVFXVFlGV2x5aEdOOEpmWk94Z1VuUmdzUW95Ly9tZlBBTU9McFpaQWtRUG9Lc2pLUUVDcXZiOFF5ekFJVWRkYzNDTmRjU2JJYkxCM3hxc2xSbU5MYkdPbk51cDZ1NCtCOGNKVEVUZHkwNU5KZjRkWE9nL1p1d24xWVhlS25qSlEva3RXblQvN05hQzdaZmlZUW14Y2ZWRkRScFhlVXpPcUdldk83TTNXYWNVWHViV2RieTdTZzdFNm50Tm54YU1GeHV0a2o5N01ReDRJUXZRM0JacjR4MTJqSy9XL1FIbFJoMWIvMFkvbTh0c0xYaVBqYnBHU2p4NzE0d0REKzJiQWlNbjZyZG0yNlBPUHpXRUlKNHFRRWxSS0FqWW9KWGZXNDlRTVNKUlJLY3NqR204YXBZcDJUclpJelJCcGdDUFp1cXdrb01xVy9qTGpWOGc2M3p2bzRweWNSMVFjK3JObndBWnpCbEFqQXRydm5rSWJlUFBqcmZpYjFneWd4NlMrYUNyRWZGd0tDTG1hdHhhNUkrUmJieUFwYTJZemdjUC93cEhsb1dBQTBDTVFDV2poS2hlSFl5K2FDRTlDbURFTURZRDE5WWg0cDF3dkRlVUVBYWhTYVFhcGJOM3hrVlZ6cXZFQ0hYeklUQVZsaz0iLCJjaXBoZXJUZXh0IjoiQVFJQkFIakRycGgrUDFTSnB1M2VJdmJ1V1IveGxkNkxzMTY4RFNyWW16OWpKdHZQVEFHY0JZVUpheXFqU0RKQStEcWp4NXVKQUFBQW9qQ0Jud1lKS29aSWh2Y05BUWNHb0lHUk1JR09BZ0VBTUlHSUJna3Foa2lHOXcwQkJ3RXdIZ1lKWUlaSUFXVURCQUV1TUJFRURDdmlieUVIUzMxVEFLZW9yZ0lCRUlCYlQ3T0xOWXk1QThmekkzZkRBdFppQzh6eEJ2VU5zeFQ1VFIzY2VnSHB4T1VQYVhnQmFmcU5HQVVuM0xQYXEvVTcwbXJuNVJzQmdMTGljdlM5MUZiVy9ESEdmUUl5WDE4T1FaNEtVRWh6d

[07/21/26 23:40:34] INFO     Final Resource Status: Succeeded                                     ]8;id=3563668;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=3563669;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#1310\1310]8;;\


Final Status: Succeeded


In [11]:
# show results
execution.show_results()

[07/21/26 23:40:35] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=3563674;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=3563675;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/21/26 23:40:36] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=3563680;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=3563681;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     S3 bucket: notebook-test-engine-ds-674622101542-usw2,        ]8;id=3563688;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py\show_results_utils.py]8;;\:]8;id=3563689;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py#129\129]8;;\
                             prefix: output/custom-scorer-eval                                                     

                    INFO     Extracted training job name:                                  ]8;id=3563695;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py\show_results_utils.py]8;;\:]8;id=3563696;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py#66\66]8;;\
                             pipelines-fg3gkadt11wc-EvaluateCustomModel-zoonTwz5R1 from                            
                             step: EvaluateCustomModel                                                             

[07/21/26 23:40:37] INFO     Searching for results_*.json in                              ]8;id=3563702;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py\show_results_utils.py]8;;\:]8;id=3563703;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py#152\152]8;;\
                             s3://notebook-test-engine-ds-674622101542-usw2/output/custom                          
                             -scorer-eval/pipelines-fg3gkadt11wc-EvaluateCustomModel-zoon                          
                             Twz5R1/output/output/                                                                 

                    INFO     Found results file:                                          ]8;id=3563709;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py\show_results_utils.py]8;;\:]8;id=3563710;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py#171\171]8;;\
                             output/custom-scorer-eval/pipelines-fg3gkadt11wc-EvaluateCus                          
                             tomModel-zoonTwz5R1/output/output/eval-meta_textgeneration_l                          
                             lama_3_2-g03qe/eval_results/results_2026-07-21T23-39-32.1965                          
                             54+00-00.json                                                                         

                    INFO     Using metrics from key: 'custom|gen_qa_gen_qa|0' (gen_qa or   ]8;id=3563716;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py\show_results_utils.py]8;;\:]8;id=3563717;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py#99\99]8;;\
                             custom_scorer format)                                                                 

                Custom Model Results                
╭────────────────────────────────┬─────────────────╮
│ Metric                         │           Value │
├────────────────────────────────┼─────────────────┤
│ bleu                           │           0.00% │
│ bleu_stderr                    │          0.0000 │
│ byoc_failure_count             │           0.00% │
│ em                             │           0.00% │
│ em_stderr                      │          0.0000 │
│ f1                             │           0.00% │
│ f1_score_quasi                 │           0.00% │
│ f1_score_quasi_stderr          │          0.0000 │
│ f1_stderr                      │          0.0000 │
│ prime_math                     │           0.00% │
│ qem                            │           0.00% │
│ qem_stderr                     │          0.0000 │
│ rouge1                         │           0.00% │
│ rouge1_stderr                  │          0.0000 │
│ rouge2                         │           0.00% │
│ rouge2_stderr                  │          0.0000 │
│ rougeL                         │           0.00% │
│ rougeL_stderr                  │          0.0000 │
╰────────────────────────────────┴─────────────────╯

╭─────────────────────────────────────────── Result Artifacts Location ───────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│  📦 Full evaluation artifacts available at:                                                                     │
│                                                                                                                 │
│  Custom Model:                                                                                                  │
│    s3://notebook-test-engine-ds-674622101542-usw2/output/custom-scorer-eval/pipelines-fg3gkadt11wc-EvaluateCus  │
│  tomModel-zoonTwz5R1/output/output/None/eval_results/                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Retrieve Existing Job

You can retrieve a previously started evaluation job using its ARN.

In [12]:
from sagemaker.train.evaluate import EvaluationPipelineExecution

# Get existing job by ARN
existing_arn = execution.arn  # Or use a specific ARN

existing_exec = EvaluationPipelineExecution.get(arn=existing_arn)

print(f"Retrieved job: {existing_exec.name}")
print(f"Status: {existing_exec.status.overall_status}")

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=3563722;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=3563723;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/21/26 23:40:38] INFO     Extracted s3_output_path from training job                            ]8;id=3563729;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=3563730;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#430\430]8;;\
                             pipelines-fg3gkadt11wc-EvaluateCustomModel-zoonTwz5R1:                                
                             s3://notebook-test-engine-ds-674622101542-usw2/output/custom-scorer-e                 
                             val/                                                                                  

Retrieved job: fg3gkadt11wc
Status: Succeeded


## List All Custom Scorer Evaluations

Retrieve all custom scorer evaluation executions.

In [13]:
# Get all custom scorer evaluations
all_executions = list(CustomScorerEvaluator.get_all())

print(f"Found {len(all_executions)} custom scorer evaluation(s):\n")
for execution in all_executions:
    print(f"  - {execution.name} - {execution.arn}: {execution.status.overall_status}")

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=3563735;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=3563736;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/21/26 23:40:39] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=3563741;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=3563742;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/21/26 23:40:40] INFO     Extracted s3_output_path from training job                            ]8;id=3563747;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=3563748;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#430\430]8;;\
                             pipelines-fg3gkadt11wc-EvaluateCustomModel-zoonTwz5R1:                                
                             s3://notebook-test-engine-ds-674622101542-usw2/output/custom-scorer-e                 
                             val/                                                                                  

Found 1 custom scorer evaluation(s):

  - fg3gkadt11wc - arn:aws:sagemaker:us-west-2:674622101542:pipeline/SagemakerEvaluation-CustomScorerEvaluation-91d7d4da-03ce-4c62-aaa6-aa69a81b2b82/execution/fg3gkadt11wc: Succeeded


## Stop a Running Job (Optional)

You can stop a running evaluation if needed.

In [14]:
# Uncomment to stop the job
# execution.stop()
# print(f"Execution stopped. Status: {execution.status.overall_status}")

## Summary

This notebook demonstrated:
1. ✅ Creating a CustomScorerEvaluator with a custom evaluator ARN
2. ✅ Starting an evaluation job
3. ✅ Monitoring job progress with refresh() and wait()
4. ✅ Retrieving existing jobs
5. ✅ Listing all custom scorer evaluations

### Key Points:
- The `evaluator` parameter accepts:
  - Custom evaluator ARN (for AI Registry evaluators)
  - Built-in metric names ("code_executions", "math_answers", "exact_match")
  - Evaluator objects from sagemaker.ai_registry.evaluator.Evaluator
- Set `evaluate_base_model=False` to only evaluate the custom model
- Use `execution.wait()` for automatic monitoring with rich visual feedback
- Use `execution.refresh()` for manual status updates
- The SageMaker session is automatically inferred from your environment